This notebook demonstrates how to run the forest deforestation User Defined Process

In [14]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [15]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [16]:
connection.authenticate_oidc()

INFO:openeo.rest.connection:Found OIDC providers: ['CDSE']
INFO:openeo.rest.connection:No OIDC provider given, but only one available: 'CDSE'. Using that one.
INFO:openeo.rest.connection:Using default client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e' from OIDC provider 'CDSE' info.
INFO:openeo.rest.connection:Found refresh token: trying refresh token based authentication.
INFO:openeo.rest.auth.oidc:Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')
INFO:openeo.rest.connection:Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
INFO:openeo.rest.auth.config:Storing refresh token for issuer 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE' (client 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')


Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [28]:
job = connection.job("j-26111910445148efb6496c732b470cfe")
print(job.status())
meta = job.get_results().get_metadata()
print(meta.get("type"), meta.keys())

features = meta.get("features") or [meta]
for item in features:
    print(item.get("bbox"))
    props = item.get("properties", {})
    print(props.get("datetime"), props.get("start_datetime"), props.get("end_datetime"))
    print(list(item.get("assets", {})))

finished
Collection dict_keys(['assets', 'description', 'extent', 'id', 'license', 'links', 'openeo:status', 'providers', 'stac_extensions', 'stac_version', 'summaries', 'type'])
None
None None None
['openEO.tif']


In [29]:
import json
print(json.dumps(meta["extent"], indent=2))
for link in meta["links"]:
    print(link.get("rel"), link.get("title"), link.get("href", "")[:160])
print(json.dumps(meta["assets"]["openEO.tif"], indent=2)[:2000])

{
  "spatial": {
    "bbox": [
      [
        30.55,
        1.07,
        31.23,
        1.55
      ]
    ]
  },
  "temporal": {
    "interval": [
      [
        "2019-10-01T00:00:00Z",
        "2025-04-01T00:00:00Z"
      ]
    ]
  }
}
derived_from Derived from /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2020/06/30/S1A_IW_GRDH_1SDV_20200630T161322_20200630T161347_033248_03DA26_C5FD_COG.SAFE /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2020/06/30/S1A_IW_GRDH_1SDV_20200630T161322_20200630T161347_033248_03DA26_C5FD_COG.SAFE
derived_from Derived from /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2020/03/02/S1A_IW_GRDH_1SDV_20200302T161317_20200302T161342_031498_03A0B5_465B_COG.SAFE /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2020/03/02/S1A_IW_GRDH_1SDV_20200302T161317_20200302T161342_031498_03A0B5_465B_COG.SAFE
derived_from Derived from /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2020/01/14/S1A_IW_GRDH_1SDV_20200114T161318_20200114T161343_030798_038860_AD74_COG.SAFE /eodata/Sentinel-1/SAR/IW_GRDH_1S-COG/2020/01/14/S1A

In [17]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

resample_spatial_resolution = 30  # m

temporal_variability_threshold = 0.5  # units: dB
flattening_threshold = 0.12  # units: dimensionless
logistic_sse_threshold = 18.3  # units: dB^2

min_connected_area = 10000  # m^2

In [18]:
# more restrictive (excludes more pixels from being detected as deforestation) than default
temporal_variability_threshold = 0.6
flattening_threshold = 0.14

## Sentinel 1

In [19]:
# load results from previous batch job
JOB_ID = "j-26111910445148efb6496c732b470cfe"  # change job, take from s1 processing

s1_dB = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
    bands=["sd", "p05", "p95", "min_sse", "min_sse_t"],
)

In [20]:
# fix for nodata tiling artifacts
# https://forum.dataspace.copernicus.eu/t/nodata-tiling-artifacts/5732/6

s1_dB.result_node().update_arguments(featureflags={"retain_nodata_tiles": True})

In [21]:
# load_stac adds a time dimension 😠
s1_dB = s1_dB.drop_dimension("t")

# load forest baseline

In [30]:
# load results from previous batch job
JOB_ID = "j-261119103751464998796dca74b19409"  # change job, take from s1 processing

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
    # note: important to specify temporal_extent here
    # otherwise openEO writes STAC temporal interval [null, null]
    temporal_extent="2020-01-01",
    bands=["B0"],
)

In [31]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

# Run deforestation UDP

In [32]:
cube = connection.datacube_from_process(
    "deforestation",
    namespace=urls.DEFORESTATION_UDP,
    # spatial_extent=spatial_extent,
    forest_baseline_datacube=forest_baseline_mask,
    sentinel_1_datacube=s1_dB,
    temporal_variability_threshold=temporal_variability_threshold,
    flattening_threshold=flattening_threshold,
)

In [33]:
job = cube.create_job(out_format="GTiff")
# job = cube.create_job(out_format="netCDF")

In [34]:
job.start_and_wait()

0:00:00 Job 'j-2611191622194ac5aec88fef5b86d992': send 'start'
0:00:04 Job 'j-2611191622194ac5aec88fef5b86d992': created (progress 0%)
0:00:10 Job 'j-2611191622194ac5aec88fef5b86d992': queued (progress 0%)
0:00:16 Job 'j-2611191622194ac5aec88fef5b86d992': queued (progress 0%)
0:00:24 Job 'j-2611191622194ac5aec88fef5b86d992': queued (progress 0%)
0:00:34 Job 'j-2611191622194ac5aec88fef5b86d992': queued (progress 0%)
0:00:46 Job 'j-2611191622194ac5aec88fef5b86d992': queued (progress 0%)
0:01:02 Job 'j-2611191622194ac5aec88fef5b86d992': running (progress N/A)
0:01:21 Job 'j-2611191622194ac5aec88fef5b86d992': running (progress N/A)
0:01:45 Job 'j-2611191622194ac5aec88fef5b86d992': running (progress N/A)
0:02:15 Job 'j-2611191622194ac5aec88fef5b86d992': running (progress N/A)
0:02:52 Job 'j-2611191622194ac5aec88fef5b86d992': running (progress N/A)
0:03:39 Job 'j-2611191622194ac5aec88fef5b86d992': running (progress N/A)
0:04:37 Job 'j-2611191622194ac5aec88fef5b86d992': finished (progress 100

<BatchJob job_id='j-2611191622194ac5aec88fef5b86d992'>

In [35]:
results = job.get_results()

In [36]:
!mkdir -p output-udp/
!rm -r output-udp/

In [37]:
results.download_files("output-udp/")

INFO:openeo.rest.job:Downloading job result asset 'openEO.tif' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-2611191622194ac5aec88fef5b86d992/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=1844b5e12362498d96cf7bc2cebcc6b2%2F20261119%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20261119T170832Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYxMTE5MTYyMjE5NGFjNWFlYzg4ZmVmNWI4NmQ5OTIiXSwidXNlcl9pZCI6WyIxM2YyYTg1Mi00YzFjLTQ4N2EtOGExYS0xYzA1YjE2M2I4YTIiXX0sInRyYW5zaXRpdmVfdGFnX2tleXMiOlsidXNlcl9pZCIsImpvYl9pZCJdfSwiaXNzIjoic3RzLndhdzMtMS5vcGVuZW8udjEuZGF0YXNwYWNlLmNvcGVybmljdXMu

[PosixPath('output-udp/openEO.tif'), PosixPath('output-udp/job-results.json')]

In [38]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)